In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: c:\Users\praya\Desktop\Tarvel_Agent\trip_planner


In [2]:
from config.settings import settings

print("Weather Provider:", settings.weather_provider)
print("Weather MCP URL:", settings.weather_mcp_server_url)

print("Flight MCP URL:", settings.kiwi_mcp_server_url)
print("Agentorist MCP URL:", settings.agentorist_mcp_server_url)

Weather Provider: livedatalink
Weather MCP URL: https://livedatalink.ai/mcp
Flight MCP URL: https://mcp.kiwi.com
Agentorist MCP URL: https://mcp.agentorist.com/mcp


In [3]:
from config.settings import settings

print(type(settings.weather_provider))
print(type(settings.weather_mcp_server_url))

assert isinstance(settings.weather_provider, str)
assert isinstance(settings.weather_mcp_server_url, str)

print("Settings Validation Passed")

<class 'str'>
<class 'str'>
Settings Validation Passed


In [4]:
from config.settings import settings

assert settings.weather_provider != ""
assert settings.weather_mcp_server_url != ""

print("Environment Variables Loaded Successfully")

Environment Variables Loaded Successfully


In [5]:
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

from config.settings import settings

async def test_weather_connection():
    try:
        async with streamable_http_client(
            settings.weather_mcp_server_url
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(
                read_stream,
                write_stream
            ) as session:
                await session.initialize()

                print("Connected Successfully")

                tools = await session.list_tools()

                print(f"Found {len(tools.tools)} tools")

                for tool in tools.tools:
                    print("-", tool.name)

    except Exception as exc:
        print("Connection Failed")
        print(type(exc))
        print(exc)

await test_weather_connection()

Connected Successfully
Found 177 tools
- property_lookup
- property_search_owner
- property_search_area
- property_value_history
- fmcsa_carrier_lookup
- fmcsa_carrier_search
- fmcsa_safety_scores
- fmcsa_carrier_authority
- fmcsa_carrier_compare
- stock_quote
- stock_quote_batch
- options_chain
- stock_history
- company_info
- stock_compare
- weather_current
- weather_forecast
- air_quality
- crypto_price
- crypto_compare
- crypto_trending
- crypto_info
- package_track
- local_search
- vin_decode
- vehicle_recalls
- sanctions_screen_entity
- sanctions_screen_batch
- sanctions_get_changes
- sanctions_screen_address
- sanctions_get_entity
- sanctions_search_alias
- sanctions_status_summary
- disaster_declarations
- nfip_flood_claims
- earthquake_recent
- nws_active_alerts
- flood_zone_lookup
- hurricane_tracker
- disaster_history_summary
- court_case_search
- court_docket_lookup
- court_opinion_search
- court_judge_lookup
- court_citation_resolver
- court_oral_argument_search
- court_re

In [6]:
import asyncio
from pprint import pprint

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

from config.settings import settings

async def discover_weather_tools():
    async with streamable_http_client(
        settings.weather_mcp_server_url
    ) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream
        ) as session:
            await session.initialize()

            tools = await session.list_tools()

            for tool in tools.tools:
                print("=" * 80)
                print("TOOL:", tool.name)

                if tool.description:
                    print("\nDESCRIPTION:")
                    print(tool.description[:500])

                print("\nREQUIRED:")
                pprint(tool.inputSchema.get("required"))

                print("\nPROPERTIES:")
                pprint(tool.inputSchema.get("properties"))
                print("=" * 80)

await discover_weather_tools()

TOOL: property_lookup

DESCRIPTION:
Look up real estate property data by street address or account number. Returns property owner name, assessed value, market value, land value, improvement value, year built, square footage, lot size, acreage, exemptions (homestead, over 65, disabled veteran), and legal description. Use this for questions like "who owns this house?", "how much is this property worth?", "what's the tax value of this address?", "what are the property details?", or any real estate lookup. Coverage note: currently dem

REQUIRED:
['query']

PROPERTIES:
{'county': {'default': 'montgomery',
            'description': 'County name (default: montgomery)',
            'type': 'string'},
 'query': {'description': 'Street address or appraisal district account number',
           'minLength': 1,
           'type': 'string'}}
TOOL: property_search_owner

DESCRIPTION:
Search for real estate properties by owner name. Find all properties owned by a person, family, trust, LLC, or compan

In [8]:
async def show_weather_tools():
    async with streamable_http_client(settings.weather_mcp_server_url) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()

            for tool in tools.tools:
                if tool.name in [
                    "weather_current",
                    "weather_forecast",
                    "air_quality"
                ]:
                    print("=" * 80)
                    print(tool.name)
                    print(tool.inputSchema)

await show_weather_tools()

weather_current
{'type': 'object', 'properties': {'location': {'type': 'string', 'description': "City, zip code, or place name (e.g., 'Houston, TX', '77001', 'Paris')"}}, 'required': ['location'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}
weather_forecast
{'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'City, zip code, or place name'}, 'days': {'type': 'integer', 'minimum': 1, 'maximum': 16, 'description': 'Forecast days (default: 7)'}}, 'required': ['location'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}
air_quality
{'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'City, zip code, or place name'}}, 'required': ['location'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}


In [10]:
async with streamable_http_client(settings.weather_mcp_server_url) as (read_stream, write_stream, _):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        tools = await session.list_tools()

for tool in tools.tools:
    print(tool.name)

property_lookup
property_search_owner
property_search_area
property_value_history
fmcsa_carrier_lookup
fmcsa_carrier_search
fmcsa_safety_scores
fmcsa_carrier_authority
fmcsa_carrier_compare
stock_quote
stock_quote_batch
options_chain
stock_history
company_info
stock_compare
weather_current
weather_forecast
air_quality
crypto_price
crypto_compare
crypto_trending
crypto_info
package_track
local_search
vin_decode
vehicle_recalls
sanctions_screen_entity
sanctions_screen_batch
sanctions_get_changes
sanctions_screen_address
sanctions_get_entity
sanctions_search_alias
sanctions_status_summary
disaster_declarations
nfip_flood_claims
earthquake_recent
nws_active_alerts
flood_zone_lookup
hurricane_tracker
disaster_history_summary
court_case_search
court_docket_lookup
court_opinion_search
court_judge_lookup
court_citation_resolver
court_oral_argument_search
court_recent_filings
cve_lookup
cve_search_by_vendor
cve_search_by_keyword
cwe_lookup
epss_score
kev_status_check
cve_recent
college_search
c

In [12]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

from config.settings import settings

async def test_weather_forecast():

    async with streamable_http_client(
        settings.weather_mcp_server_url
    ) as (
        read_stream,
        write_stream,
        _
    ):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "weather_forecast",
                {
                    "location": "Mumbai",
                    "days": 7
                }
            )

            print(result)

await test_weather_forecast()

meta=None content=[TextContent(type='text', text='7-DAY FORECAST - Mumbai, Maharashtra, India\n════════════════════════════════════════════════════════\n\nDate        Hi    Lo    Conditions            Rain%  Wind\n────────────────────────────────────────────────────────\nToday       94°   86°   Thunderstorm          78%    9 mph\nTomorrow    92°   85°   Thunderstorm          59%    9 mph\nMon, Jun 1  92°   85°   Thunderstorm          57%    8 mph\nTue, Jun 2  93°   84°   Thunderstorm          63%    9 mph\nWed, Jun 3  90°   81°   Thunderstorm          80%    11 mph\nThu, Jun 4  89°   81°   Thunderstorm          77%    12 mph\nFri, Jun 5  89°   81°   Thunderstorm          73%    10 mph\n\nSunrise: 00:30  |  Sunset: 13:41', annotations=None, meta=None)] structuredContent=None isError=False


In [13]:
async def test_current_weather():

    async with streamable_http_client(
        settings.weather_mcp_server_url
    ) as (
        read_stream,
        write_stream,
        _
    ):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "weather_current",
                {
                    "location": "Mumbai"
                }
            )

            print(result)

await test_current_weather()

meta=None content=[TextContent(type='text', text='CURRENT WEATHER - Mumbai, Maharashtra, India\n════════════════════════════════════════════════════════\n\nConditions:            Thunderstorm\nTemperature:           90°F\nFeels Like:            98°F\nHumidity:              66%\nWind:                  8 mph WSW\nWind Gusts:            22 mph\nCloud Cover:           5%\nPressure:              1006.1 hPa\nUV Index:              2.3 (Moderate)\nVisibility:            15.9 miles', annotations=None, meta=None)] structuredContent=None isError=False


In [17]:
from pprint import pprint

print("=" * 80)
print("TEST 1: SETTINGS")
print("=" * 80)

from config.settings import settings

print("Weather Provider:", settings.weather_provider)
print("Weather MCP URL:", settings.weather_mcp_server_url)

print("\n")

print("=" * 80)
print("TEST 2: MCP CLIENT")
print("=" * 80)

try:
    from tools.weather_mcp_client import (
        get_current_weather,
        get_weather_forecast,
        get_air_quality,
    )

    current = get_current_weather("Mumbai")
    print("Current Weather Status:", current.get("status"))

    forecast = get_weather_forecast(
        "Mumbai",
        days=7,
    )
    print("Forecast Status:", forecast.get("status"))

    air = get_air_quality("Mumbai")
    print("Air Quality Status:", air.get("status"))

    print("✅ MCP Client Passed")

except Exception as e:
    print("❌ MCP Client Failed")
    print(type(e).__name__)
    print(e)

print("\n")

print("=" * 80)
print("TEST 3: WEATHER TOOL")
print("=" * 80)

try:
    from tools.weather_tools import get_weather

    weather = get_weather(
        destination="Mumbai",
        event_date="2026-08-15",
    )

    print("Provider:", weather.get("provider"))
    print("Status:", weather.get("status"))

    print("\nForecast Present:", "forecast" in weather)
    print("Air Quality Present:", "air_quality" in weather)

    print("✅ Weather Tool Passed")

except Exception as e:
    print("❌ Weather Tool Failed")
    print(type(e).__name__)
    print(e)

print("\n")

print("=" * 80)
print("TEST 4: WEATHER AGENT")
print("=" * 80)

try:
    from agents.weather_agent import weather_agent

    state = {
        "destination": "Mumbai",
        "event_date": "2026-08-15",
        "errors": [],
    }

    result = weather_agent(state)

    print("Weather Details Present:",
          "weather_details" in result)

    print("Weather Notes Present:",
          "weather_notes" in result)

    print("\nWeather Notes:")
    print(result.get("weather_notes"))

    print("✅ Weather Agent Passed")

except Exception as e:
    print("❌ Weather Agent Failed")
    print(type(e).__name__)
    print(e)

print("\n")

print("=" * 80)
print("FINAL RESULT")
print("=" * 80)

print("""
Checklist:

✅ Settings Loaded
✅ MCP Client Working
✅ Weather Tool Working
✅ Weather Agent Working

If all four pass:
LiveDataLink Weather MCP integration is complete.
""")

TEST 1: SETTINGS
Weather Provider: livedatalink
Weather MCP URL: https://livedatalink.ai/mcp


TEST 2: MCP CLIENT
Current Weather Status: success
Forecast Status: success
Air Quality Status: success
✅ MCP Client Passed


TEST 3: WEATHER TOOL
Provider: livedatalink
Status: success

Forecast Present: True
Air Quality Present: True
✅ Weather Tool Passed


TEST 4: WEATHER AGENT
MCP ERROR DETAILS
  + Exception Group Traceback (most recent call last):
  |   File "c:\Users\praya\Desktop\Tarvel_Agent\trip_planner\tools\weather_mcp_client.py", line 215, in get_air_quality
  |     response = _run_coroutine(_call_tool("air_quality", payload, DEFAULT_TIMEOUT_SECONDS))
  |   File "c:\Users\praya\Desktop\Tarvel_Agent\trip_planner\tools\weather_mcp_client.py", line 69, in _run_coroutine
  |     return asyncio.run(coro)
  |            ~~~~~~~~~~~^^^^^^
  |   File "C:\Users\praya\AppData\Local\Programs\Python\Python313\Lib\asyncio\runners.py", line 195, in run
  |     return runner.run(main)
  |       

In [21]:
from agents.supervisor_agent import supervisor_agent

state = {
    "destination": "new york",
    "venue": "Miami",
    "event_date": "08/06/2026",
    "errors": []
}

result = supervisor_agent(state)

print("Status:", result.get("status"))

print("\nFlight Notes:")
print(result.get("flight_notes"))

print("\nHotel Notes:")
print(result.get("hotel_notes"))

print("\nWeather Notes:")
print(result.get("weather_notes"))

print("\nSupervisor Notes:")
print(result.get("supervisor_notes"))

Agentorist MCP payload for search: {'vertical': 'local', 'query': 'restaurants', 'location': 'new york', 'agent_client': 'TripPlanner'}
Status: failed

Flight Notes:
Flight search failed.

Hotel Notes:
Based on the search results, here are some nearby restaurants and local spots in New York:

1. La Grande Boucherie - a French restaurant with a rating of 4.5 and a price range of $$$.
2. Da Andrea - Greenwich Village - an Italian restaurant with a rating of 4.4 and a price range of $$.
3. Her Name Is Han - a Korean restaurant with a rating of 4.3 and a price range of $$.
4. Kong Sihk Tong 港食堂 - a Chinese restaurant with a rating of 4.2 and a price range of $$.
5. Soothr - a Thai restaurant with a rating of 4.5 and a price range of $$.
6. Boucherie West Village - a French restaurant with a rating of 4.5 and a price range of $$$.
7. Wah Fung No 1 - a hot dog and Cantonese restaurant with a rating of 4.4 and a price range of $.
8. Mei Lai Wah Bakery - a bakery and dim sum restaurant with a 

In [16]:
from tools.weather_mcp_client import get_current_weather

print(type(get_current_weather("Mumbai")))

<class 'dict'>
